# GEPA Prompt Optimizer — Bot de triage de reclamos

**Caso de uso:** un bot de soporte recibe reclamos de clientes y debe responder siguiendo un formato estructurado estricto:

```
Severidad: CRÍTICO | ALTO | MEDIO | BAJO
Área: Facturación | Soporte Técnico | Retención | Acceso
Acción inmediata: [qué debe hacer el equipo interno]
Respuesta al cliente: [mensaje a enviar al cliente]
```

Con el seed prompt `"Eres un agente de atención al cliente."`, el bot responde con texto libre y conversacional — ignora el formato, no clasifica severidad, no distingue áreas. GEPA aprende el formato y las reglas de clasificación a partir de los ejemplos que fallan.

El dataset tiene 4 tipos de reclamos (facturación, soporte técnico, retención, acceso) con 2 casos cada uno. Cada `Dataset` incluye el perfil del cliente en `context`.

# Instalation

In [ ]:
!pip install alquimia-fair-forge[prompt-optimizer] langchain-groq

## Setup

In [1]:
import os
import logging
import getpass

logging.getLogger("httpx").setLevel(logging.WARNING)

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parents[3]))
sys.path.insert(0, str(Path().resolve().parents[3] / "examples"))

## 1. Retriever

Carga las preguntas de soporte y sus respuestas esperadas. Cada `Dataset` incluye el fragmento de FAQ relevante en `context` — el mismo que se inyecta al agente en cada consulta.

In [3]:
import json
from pathlib import Path
from fair_forge import Retriever
from fair_forge.schemas import Dataset

class SupportRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        data_path = Path("../data/dataset.json")
        with open(data_path, encoding="utf-8") as f:
            return [Dataset.model_validate(item) for item in json.load(f)]

c:\Users\ericf\Desktop\Alquimia\fair-forge\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 2. Model

El modelo que GEPA usa para **generar prompts candidatos mejorados**. No es el agente que se está optimizando — ese usa el mismo modelo con el prompt candidato como system prompt.

In [4]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")

## 3. Run the optimizer

- `seed_prompt`: el prompt actual del agente — deliberadamente vago para que GEPA tenga margen de mejora.
- `objective`: describe en lenguaje natural qué hace un buen agente de soporte. GEPA lo usa para evaluar respuestas y guiar la generación de mejores prompts.
- `iterations`: cuántas rondas de mejora puede hacer GEPA como máximo. En cada ronda toma el mejor prompt de la ronda anterior, identifica los ejemplos donde falla y genera nuevos candidatos.
- `candidates_per_iteration`: cuántos prompts candidatos genera GEPA por ronda. Evalúa todos y se queda con el mejor. Más candidatos = más chances de mejorar, pero más llamadas al LLM.

GEPA se detiene antes de agotar las iteraciones si ningún candidato supera el score actual — es decir, cuando el prompt ya convergió.

In [5]:
from fair_forge.prompt_optimizer.gepa import GEPAOptimizer

result = GEPAOptimizer.run(
    retriever=SupportRetriever,
    model=model,
    seed_prompt="Eres un agente de atención al cliente.",
    objective=(
        "Clasificar cada reclamo según severidad (CRÍTICO, ALTO, MEDIO, BAJO) y área (Facturación, Soporte Técnico, Retención, Acceso). "
        "Responder siempre en el siguiente formato exacto:\n"
        "Severidad: <nivel>\nÁrea: <área>\nAcción inmediata: <qué debe hacer el equipo>\nRespuesta al cliente: <mensaje para el cliente>\n"
        "La respuesta al cliente debe ser directa, empática y no inventar información que no esté en el contexto."
    ),
    iterations=5,
    candidates_per_iteration=3,
)

Evaluating seed prompt on 8 examples...


Optimizing prompt:   0%|          | 0/5 [00:00<?, ?it/s]

## 4. Results

In [6]:
print(f"Initial score : {result.initial_score:.2f} / 1.00  ({result.n_examples} examples)")
print(f"Final score   : {result.final_score:.2f} / 1.00  ({result.n_examples} examples)")
print(f"Iterations run: {result.iterations_run}")
print()

if result.history:
    print("Score per iteration:")
    for it in result.history:
        candidates_scores = "  |  ".join(f"{c.score:.2f}" for c in it.candidates)
        print(f"  Iteration {it.iteration}: [{candidates_scores}]  → best: {it.best_score:.2f}")
    print()

print("=== Seed prompt ===")
print("Eres un agente de atención al cliente.")
print()
print("=== Optimized prompt ===")
print(result.optimized_prompt)

Initial score : 0.38 / 1.00  (8 examples)
Final score   : 0.56 / 1.00  (8 examples)
Iterations run: 3

Score per iteration:
  Iteration 1: [0.38  |  0.30  |  0.47]  → best: 0.47
  Iteration 2: [0.33  |  0.44  |  0.56]  → best: 0.56
  Iteration 3: [0.38  |  0.33  |  0.38]  → best: 0.38

=== Seed prompt ===
Eres un agente de atención al cliente.

=== Optimized prompt ===
Analiza cada reclamo y responde con un formato estándar que incluye la severidad, área, acción inmediata y respuesta al cliente, asegurándote de que cada respuesta sea concisa, directa y se ajuste a las necesidades específicas del cliente.
